<a href="https://colab.research.google.com/github/miruts-code/nanoGpt/blob/main/Copy_of_nanoGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

saving student model metrics

In [37]:
distillation_metrics = {
    "technique": "distillation_toy_4layer_256dim",
    "param_count": params_total_d,
    "model_size_mb": size_mb_d,
    "perplexity_mean": ppl_d,
    "average_loss": avg_loss_d,
    "latency_mean_ms": latency_mean_d,
    "latency_std_ms": latency_std_d,
    "macs_per_forward": macs_per_forward_d,
    "sample_output": sample_output_d,
    "notes": "Toy-scale knowledge distillation: 4-layer/256-dim student trained from random init for 200 steps on WikiText tokens, using KL-divergence (temperature=2.0) + hard-label loss (alpha=0.5) against the base GPT-2 teacher. Not representative of full distillation quality — scoped down for time. Hard loss dropped from ~10.8 to ~7.9 over training, confirming real learning occurred."
}

save_path_d = os.path.join(RESULTS_DIR, "distillation_metrics.json")
with open(save_path_d, "w") as f:
    json.dump(distillation_metrics, f, indent=2)

print(f"Saved distillation metrics to {save_path_d}")
print(json.dumps(distillation_metrics, indent=2))

Saved distillation metrics to /content/drive/MyDrive/nanoGPT-project/results/distillation_metrics.json
{
  "technique": "distillation_toy_4layer_256dim",
  "param_count": 16287488,
  "model_size_mb": 62.15055179595947,
  "perplexity_mean": 2093.6396782891024,
  "average_loss": 7.64665930321876,
  "latency_mean_ms": 0.1899341009767852,
  "latency_std_ms": 0.011386067085130673,
  "macs_per_forward": 821121280,
  "sample_output": "The history of artificial intelligence for video , from on only and both both the,, over\n for the \" - and\n for the from ( , , in the the, ( for over the\n A a Chinese ( and\n at in ,\n the in the and the",
  "notes": "Toy-scale knowledge distillation: 4-layer/256-dim student trained from random init for 200 steps on WikiText tokens, using KL-divergence (temperature=2.0) + hard-label loss (alpha=0.5) against the base GPT-2 teacher. Not representative of full distillation quality \u2014 scoped down for time. Hard loss dropped from ~10.8 to ~7.9 over training, c

metrics on the distilled student model

In [36]:
student.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Param count and size
params_total_d = sum(p.numel() for p in student.parameters())
torch.save(student.state_dict(), "student_temp.pt")
size_mb_d = os.path.getsize("student_temp.pt") / (1024 ** 2)
os.remove("student_temp.pt")
print(f"Student param count: {params_total_d:,}")
print(f"Student model size (on disk): {size_mb_d:.2f} MB")

# Perplexity
ppl_d, avg_loss_d = compute_perplexity(student, tokens_gpu, device=device)
print(f"Student Perplexity: {ppl_d:.3f}, Loss: {avg_loss_d:.4f}")

# Latency
latency_mean_d, latency_std_d = measure_latency(student, tokens_gpu, device=device)
print(f"Student Latency — mean: {latency_mean_d:.4f} ms/token, std: {latency_std_d:.4f} ms/token")

# MACs
x_sample_d = tokens_gpu[:256].unsqueeze(0)
flops_d = FlopCountAnalysis(student, x_sample_d)
macs_per_forward_d = flops_d.total()
print(f"Student MACs per forward pass: {macs_per_forward_d:,}")


# Sample output
prompt_tokens_gpu = torch.tensor(enc.encode("The history of artificial intelligence"), dtype=torch.long).unsqueeze(0).to(device)
with torch.no_grad():
    generated_d = student.generate(prompt_tokens_gpu, max_new_tokens=50, temperature=0.8, top_k=40)
sample_output_d = enc.decode(generated_d[0].tolist())
print(sample_output_d)

Student param count: 16,287,488
Student model size (on disk): 62.15 MB
Student Perplexity: 2093.640, Loss: 7.6467


transformer.h.0.attn.attn_dropout, transformer.h.1.attn.attn_dropout, transformer.h.2.attn.attn_dropout, transformer.h.3.attn.attn_dropout


Student Latency — mean: 0.1899 ms/token, std: 0.0114 ms/token
Student MACs per forward pass: 821,121,280
The history of artificial intelligence for video , from on only and both both the,, over
 for the " - and
 for the from ( , , in the the, ( for over the
 A a Chinese ( and
 at in ,
 the in the and the


minimal distillaion training loop

In [35]:
import torch.nn.functional as F

teacher = model  # your original base model
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(student.parameters(), lr=3e-4)
block_size = 256
n_steps = 200          # small/toy — increase later if you want better quality
temperature = 2.0
alpha = 0.5            # weight between distillation loss and hard-label loss

tokens_gpu = tokens.to(device)
losses = []

for step in range(n_steps):
    i = torch.randint(0, len(tokens_gpu) - block_size - 1, (1,)).item()
    x = tokens_gpu[i:i+block_size].unsqueeze(0)
    y = tokens_gpu[i+1:i+block_size+1].unsqueeze(0)

    with torch.no_grad():
        teacher_logits, _ = teacher(x)

    student_logits, hard_loss = student(x, y)

    soft_teacher = F.log_softmax(teacher_logits / temperature, dim=-1)
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    distill_loss = F.kl_div(soft_student, soft_teacher, log_target=True, reduction='batchmean') * (temperature ** 2)

    loss = alpha * distill_loss + (1 - alpha) * hard_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if step % 20 == 0:
        print(f"Step {step}: loss={loss.item():.4f} (distill={distill_loss.item():.4f}, hard={hard_loss.item():.4f})")

print("Distillation training complete.")
student.eval()

Step 0: loss=963.3499 (distill=1915.8671, hard=10.8326)
Step 20: loss=593.7695 (distill=1177.8923, hard=9.6467)
Step 40: loss=418.1320 (distill=827.1143, hard=9.1498)
Step 60: loss=2841.9849 (distill=5675.1904, hard=8.7793)
Step 80: loss=533.3432 (distill=1057.8438, hard=8.8427)
Step 100: loss=346.5280 (distill=684.7451, hard=8.3110)
Step 120: loss=622.1599 (distill=1236.3682, hard=7.9517)
Step 140: loss=534.4833 (distill=1061.0016, hard=7.9649)
Step 160: loss=3324.2737 (distill=6641.0400, hard=7.5071)
Step 180: loss=250.5867 (distill=493.3051, hard=7.8682)
Distillation training complete.


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 256)
    (wpe): Embedding(1024, 256)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-3): 4 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=256, out_features=768, bias=True)
          (c_proj): Linear(in_features=256, out_features=256, bias=True)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=256, out_features=1024, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1024, out_features=256, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=256, out_features=50257, bias=False)
)

define student model(knowledge distillation)

In [34]:
from model import GPT, GPTConfig

student_config = GPTConfig(
    block_size=1024,
    vocab_size=50257,
    n_layer=4,      # teacher has 12
    n_head=4,       # teacher has 12
    n_embd=256,     # teacher has 768
    bias=True,
)

student = GPT(student_config)
student.to(device)
student.train()

student_params = sum(p.numel() for p in student.parameters())
print(f"Student param count: {student_params:,} (vs teacher's {params_total:,})")

number of parameters: 16.03M
Student param count: 16,287,488 (vs teacher's 124,439,808)


save the pruning_metrics

In [33]:
pruning_metrics = {
    "technique": "pruning_unstructured_l1_30pct",
    "param_count": params_total_p,
    "model_size_mb": size_mb_p,
    "sparsity": actual_sparsity,
    "perplexity_mean": ppl_p,
    "average_loss": avg_loss_p,
    "latency_mean_ms": latency_mean_p,
    "latency_std_ms": latency_std_p,
    "macs_per_forward": macs_per_forward_p,
    "sample_output": sample_output_p,
    "notes": "Unstructured L1 magnitude pruning at 30% target sparsity per Linear layer, weights permanently zeroed via prune.remove(). Model size/latency/MACs remain similar to base since tensor shapes are unchanged — only actual model weights are zeroed, not removed. Real-world speedup would require sparse-aware hardware/kernels."
}

save_path_p = os.path.join(RESULTS_DIR, "pruning_metrics.json")
with open(save_path_p, "w") as f:
    json.dump(pruning_metrics, f, indent=2)

print(f"Saved pruning metrics to {save_path_p}")
print(json.dumps(pruning_metrics, indent=2))

Saved pruning metrics to /content/drive/MyDrive/nanoGPT-project/results/pruning_metrics.json
{
  "technique": "pruning_unstructured_l1_30pct",
  "param_count": 124439808,
  "model_size_mb": 474.7520742416382,
  "sparsity": 0.30000005990349127,
  "perplexity_mean": 70.3822499297965,
  "average_loss": 4.253941099694434,
  "latency_mean_ms": 4.686950873827733,
  "latency_std_ms": 2.4974363131885378,
  "macs_per_forward": 21806445312,
  "sample_output": "The history of artificial intelligence will be the first to know.\n\nThe first artificial intelligence research was conducted by Jules de Saint Martin in 1936, a mathematician from the University of Lyon in France, who was at the time in France, and worked in France to discover the",
  "notes": "Unstructured L1 magnitude pruning at 30% target sparsity per Linear layer, weights permanently zeroed via prune.remove(). Model size/latency/MACs remain similar to base since tensor shapes are unchanged \u2014 only actual model weights are zeroed, 

metrics on the pruned model

In [31]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_pruned.to(device)
tokens_gpu = tokens.to(device)

# Param count and size
params_total_p = sum(p.numel() for p in model_pruned.parameters())

torch.save(model_pruned.state_dict(), "pruned_temp.pt")
size_mb_p = os.path.getsize("pruned_temp.pt") / (1024 ** 2)
os.remove("pruned_temp.pt")

print(f"Pruned param count: {params_total_p:,}")
print(f"Pruned model size (on disk): {size_mb_p:.2f} MB")

# Perplexity
ppl_p, avg_loss_p = compute_perplexity(model_pruned, tokens_gpu, device=device)
print(f"Pruned Perplexity: {ppl_p:.3f}, Loss: {avg_loss_p:.4f}")

# Latency
latency_mean_p, latency_std_p = measure_latency(model_pruned, tokens_gpu, device=device)
print(f"Pruned Latency — mean: {latency_mean_p:.4f} ms/token, std: {latency_std_p:.4f} ms/token")

# MACs
x_sample_p = tokens_gpu[:256].unsqueeze(0)
flops_p = FlopCountAnalysis(model_pruned, x_sample_p)
macs_per_forward_p = flops_p.total()
print(f"Pruned MACs per forward pass: {macs_per_forward_p:,}")


# Sample output
prompt_tokens_gpu = torch.tensor(enc.encode("The history of artificial intelligence"), dtype=torch.long).unsqueeze(0).to(device)
with torch.no_grad():
    generated_p = model_pruned.generate(prompt_tokens_gpu, max_new_tokens=50, temperature=0.8, top_k=40)
sample_output_p = enc.decode(generated_p[0].tolist())
print(sample_output_p)

Pruned param count: 124,439,808
Pruned model size (on disk): 474.75 MB
Pruned Perplexity: 70.382, Loss: 4.2539
Pruned Latency — mean: 4.6870 ms/token, std: 2.4974 ms/token


transformer.h.0.attn.attn_dropout, transformer.h.1.attn.attn_dropout, transformer.h.10.attn.attn_dropout, transformer.h.11.attn.attn_dropout, transformer.h.2.attn.attn_dropout, transformer.h.3.attn.attn_dropout, transformer.h.4.attn.attn_dropout, transformer.h.5.attn.attn_dropout, transformer.h.6.attn.attn_dropout, transformer.h.7.attn.attn_dropout, transformer.h.8.attn.attn_dropout, transformer.h.9.attn.attn_dropout


Pruned MACs per forward pass: 21,806,445,312
The history of artificial intelligence will be the first to know.

The first artificial intelligence research was conducted by Jules de Saint Martin in 1936, a mathematician from the University of Lyon in France, who was at the time in France, and worked in France to discover the


make pruning permanent and check actual sparcity

In [30]:
# prune.l1_unstructured only masks weights; this bakes the zeros in permanently
for name, module in model_pruned.named_modules():
    if isinstance(module, torch.nn.Linear):
        prune.remove(module, 'weight')

# Verify actual sparsity achieved
total_params = 0
zero_params = 0
for name, module in model_pruned.named_modules():
    if isinstance(module, torch.nn.Linear):
        total_params += module.weight.numel()
        zero_params += (module.weight == 0).sum().item()

actual_sparsity = zero_params / total_params
print(f"Actual sparsity across Linear layers: {actual_sparsity:.4f} ({actual_sparsity*100:.2f}%)")

Actual sparsity across Linear layers: 0.3000 (30.00%)


Apply unstructured pruning (30%)

In [29]:
import torch.nn.utils.prune as prune

# Fresh copy so 'model' (base) and 'model_quantized' stay untouched
model_pruned = GPT.from_pretrained('gpt2')
model_pruned.eval()

# Apply L1-magnitude unstructured pruning to every Linear layer's weight
prune_amount = 0.30
for name, module in model_pruned.named_modules():
    if isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=prune_amount)

print("Pruning applied .")

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
number of parameters: 123.65M


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Pruning applied .


metrics on quantized model

In [26]:
device_cpu = 'cpu'
model_quantized.to(device_cpu)
tokens_cpu = tokens.to(device_cpu)

# Param count and size
params_total_q = sum(p.numel() for p in model_quantized.parameters())
size_mb_q = sum(p.numel() * p.element_size() for p in model_quantized.parameters()) / (1024 ** 2)
print(f"Quantized param count: {params_total_q:,}")
print(f"Quantized model size: {size_mb_q:.2f} MB")

# Perplexity
ppl_q, avg_loss_q = compute_perplexity(model_quantized, tokens_cpu, device='cpu')
print(f"Quantized Perplexity: {ppl_q:.3f}, Loss: {avg_loss_q:.4f}")

# Latency (CPU)
latency_mean_q, latency_std_q = measure_latency(model_quantized, tokens_cpu, device='cpu')
print(f"Quantized Latency — mean: {latency_mean_q:.4f} ms/token, std: {latency_std_q:.4f} ms/token")

# Sample output
prompt_tokens_cpu = prompt_tokens.to(device_cpu)
with torch.no_grad():
    generated_q = model_quantized.generate(prompt_tokens_cpu, max_new_tokens=50, temperature=0.8, top_k=40)
sample_output_q = enc.decode(generated_q[0].tolist())
print(sample_output_q)

Quantized param count: 39,422,208
Quantized model size: 150.38 MB
Quantized Perplexity: 15733.342, Loss: 9.6635
Quantized Latency — mean: 2.6953 ms/token, std: 0.6172 ms/token
The history of artificial intelligence first-develop. 2, M/D, a a "--h, on the way The "-. the first-l for a, the, a, the in. of a, for, with, " the. the I


PTQ optimization , int8

In [25]:
import torch.quantization

# Load a fresh copy of the model so the original 'model' stays untouched for reference
model_fp32 = GPT.from_pretrained('gpt2')
model_fp32.eval()

model_quantized = torch.quantization.quantize_dynamic(
    model_fp32,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("Quantization applied.")
print(model_quantized)

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
number of parameters: 123.65M


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/tmp/ipykernel_3710/1695019870.py:7: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_quantized = torch.quantization.quantize_dynamic(


Quantization applied.
GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): DynamicQuantizedLinear(in_features=768, out_features=2304, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
          (c_proj): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): DynamicQuantizedLinear(in_features=768, out_features=3072, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
          (gelu): GELU(approximate='none')
          (c_proj): DynamicQuantizedLinear(in_features=3072, out_features=768, dtype=torch.qint8, qscheme=torch.per_tenso

assemble and save base_metrics.json

In [24]:
import json
import os

base_metrics = {
    "technique": "base",
    "param_count": params_total,
    "model_size_mb": size_mb,
    "perplexity_mean": ppl,
    "average_loss": loss,
    "latency_mean_ms": latency_mean,
    "latency_std_ms": latency_std,
    "macs_per_forward": macs_per_forward,
    "sample_output": sample_output,
}

os.makedirs(RESULTS_DIR, exist_ok=True)
save_path = os.path.join(RESULTS_DIR, "base_metrics.json")

with open(save_path, "w") as f:
    json.dump(base_metrics, f, indent=2)

print(f"Saved baseline metrics to {save_path}")
print(json.dumps(base_metrics, indent=2))

Saved baseline metrics to /content/drive/MyDrive/nanoGPT-project/results/base_metrics.json
{
  "technique": "base",
  "param_count": 124439808,
  "model_size_mb": 474.74195194244385,
  "perplexity_mean": 49.0702002703826,
  "average_loss": 3.8932519314136913,
  "latency_mean_ms": 3.8693562976562568,
  "latency_std_ms": 0.7244249321893615,
  "macs_per_forward": 21806445312,
  "sample_output": "The history of artificial intelligence is not an abstract concept, but the work of scientists, engineers, entrepreneurs, and entrepreneurs. The history of artificial intelligence is a long and complex one where some may find it difficult to explain the basic concepts or methods to their own minds. And the"
}


generate a sample output from the base model

In [23]:
model.eval()
prompt = "The history of artificial intelligence"
prompt_tokens = torch.tensor(enc.encode(prompt), dtype=torch.long).unsqueeze(0).to('cuda' if torch.cuda.is_available() else 'cpu')

with torch.no_grad():
    generated = model.generate(prompt_tokens, max_new_tokens=50, temperature=0.8, top_k=40)

sample_output = enc.decode(generated[0].tolist())
print(sample_output)

The history of artificial intelligence is not an abstract concept, but the work of scientists, engineers, entrepreneurs, and entrepreneurs. The history of artificial intelligence is a long and complex one where some may find it difficult to explain the basic concepts or methods to their own minds. And the


measure peak GPU memory for one forward pass

install fvcore and measure MACs per forward pass

In [20]:
!pip install fvcore --quiet
from fvcore.nn import FlopCountAnalysis

x_sample = tokens[:256].unsqueeze(0).to('cuda' if torch.cuda.is_available() else 'cpu')
flops = FlopCountAnalysis(model, x_sample)
macs_per_forward = flops.total()
print(f"MACs per forward pass: {macs_per_forward:,}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 9.7 MB/s eta 0:00:00


transformer.h.0.attn.attn_dropout, transformer.h.1.attn.attn_dropout, transformer.h.10.attn.attn_dropout, transformer.h.11.attn.attn_dropout, transformer.h.2.attn.attn_dropout, transformer.h.3.attn.attn_dropout, transformer.h.4.attn.attn_dropout, transformer.h.5.attn.attn_dropout, transformer.h.6.attn.attn_dropout, transformer.h.7.attn.attn_dropout, transformer.h.8.attn.attn_dropout, transformer.h.9.attn.attn_dropout


MACs per forward pass: 21,806,445,312


latency funciton and its run

In [19]:
import time

def measure_latency(model, tokens, block_size=256, n_runs=20, device='cuda' if torch.cuda.is_available() else 'cpu'):
    model.to(device)
    x = tokens[:block_size].unsqueeze(0).to(device)
    with torch.no_grad():
        for _ in range(3):
            model(x)
        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            model(x)
            if device == 'cuda':
                torch.cuda.synchronize()
            times.append((time.perf_counter() - start) * 1000 / block_size)
    latency_mean = sum(times) / len(times)
    latency_std = (sum((t - latency_mean) ** 2 for t in times) / len(times)) ** 0.5
    return latency_mean, latency_std
latency_mean, latency_std = measure_latency(model, tokens)
print(f"Latency — mean: {latency_mean:.4f} ms/token, std: {latency_std:.4f} ms/token")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

Latency — mean: 3.8694 ms/token, std: 0.7244 ms/token
CUDA available: False


perplexity function and its run

In [18]:
import torch.nn.functional as F
import math

def compute_perplexity(model, tokens, block_size=256, device='cuda' if torch.cuda.is_available() else 'cpu'):
    model.to(device)
    tokens = tokens.to(device)
    total_loss = 0.0
    n_blocks = 0
    with torch.no_grad():
        for i in range(0, len(tokens) - block_size, block_size):
            x = tokens[i:i+block_size].unsqueeze(0)
            y = tokens[i+1:i+block_size+1].unsqueeze(0)
            logits, loss = model(x, y)
            total_loss += loss.item()
            n_blocks += 1
    avg_loss = total_loss / n_blocks
    perplexity = math.exp(avg_loss)
    return perplexity, avg_loss

ppl, loss = compute_perplexity(model, tokens)
print(f"Base model — Perplexity: {ppl:.3f}, Loss: {loss:.4f}")

Base model — Perplexity: 49.070, Loss: 3.8933


eval text plus tokenizing

In [17]:
!pip install datasets tiktoken --quiet
from datasets import load_dataset
import tiktoken

wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
eval_text = "\n\n".join(wikitext["text"][:200])

enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode(eval_text)
tokens = torch.tensor(tokens, dtype=torch.long)
print(f"Total eval tokens: {len(tokens):,}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Total eval tokens: 12,122


count parameters and measure checkpoint size. run it on the base model

In [16]:
def get_params_and_size(model, technique_name):
    params_total = sum(p.numel() for p in model.parameters())

    checkpoint_path = f'/content/drive/MyDrive/nanoGPT-project/checkpoints/{technique_name}.pt'
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
    torch.save(model.state_dict(), checkpoint_path)
    size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)

    return params_total, size_mb, checkpoint_path

params_total, size_mb, base_checkpoint_path = get_params_and_size(model, "base")
print(f"Total parameters: {params_total:,}")
print(f"Checkpoint size: {size_mb:.2f} MB")

Total parameters: 124,439,808
Checkpoint size: 474.74 MB


creating a results folder to save the parameters change after optimization.

In [15]:
import os
import json

RESULTS_DIR = '/content/drive/MyDrive/nanoGPT-project/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR}")

Results will be saved to: /content/drive/MyDrive/nanoGPT-project/results


confirming wether the parameters get cashed

In [14]:
import os
print(os.environ.get('HF_HOME', 'not set — using default cache'))

/content/drive/MyDrive/nanoGPT-project/hf-cache


loading the parameters from trained hugging face gpt-2 model

In [12]:
from model import GPT
import torch
model = GPT.from_pretrained('gpt2')
model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
number of parameters: 123.65M


Loading weights:   0%|          | 0/148 [00:06<?, ?it/s]

Total parameters: 124,439,808


In [13]:
!pip install torch numpy transformers datasets tiktoken tqdm pandas

In [11]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/nanoGPT-project/nanoGpt


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/nanoGPT-project/nanoGpt


In [10]:
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/nanoGPT-project/hf-cache'

In [8]:
import os
os.makedirs('/content/drive/MyDrive/nanoGPT-project', exist_ok=True)
%cd /content/drive/MyDrive/nanoGPT-project

!git clone https://github.com/miruts-code/nanoGpt.git
%cd nanoGpt

/content/drive/MyDrive/nanoGPT-project
fatal: destination path 'nanoGpt' already exists and is not an empty directory.
/content/drive/MyDrive/nanoGPT-project/nanoGpt


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
